In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
Portfolio = pd.read_csv("Portfolio_clean.csv")
transcript = pd.read_csv("transcript_cleaned_flattened.csv")

In [5]:
Portfolio.head()

,reward,social,web,mobile,email,difficulty,duration,offer_type,promo_id
0,10,1,0,1,1,10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,1,1,1,1,10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,0,1,1,1,0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,0,1,1,1,5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,0,1,0,1,20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [ ]:
transcript.head()


,customer_id,offer_id,event,amount value,Reward,time
0,1ea1a86f17234a249151ea3785925c4c,f19421c1d4aa40978ebb69ca19b0e20d,offer completed,NaN,5.0,444
1,2366b169d6de42f084c7a64c1b938f9d,f19421c1d4aa40978ebb69ca19b0e20d,offer completed,NaN,5.0,444
2,2531d3b6001a4d48bff0a91ceceb97a0,f19421c1d4aa40978ebb69ca19b0e20d,offer completed,NaN,5.0,444
3,25f66f03dbcb42f89a31b53d15a8f729,f19421c1d4aa40978ebb69ca19b0e20d,offer completed,NaN,5.0,444
4,27cb665ec6754c2b8731f37deefb229c,f19421c1d4aa40978ebb69ca19b0e20d,offer completed,NaN,5.0,444


In [10]:
transcript = transcript.rename(columns={"amount value": "amount"})

In [24]:
print("Portfolio columns:", Portfolio.columns.tolist())
print("transcript columns:", transcript.columns.tolist())
print("\nEvent counts:")
print(transcript["event"].value_counts(dropna=False))

Portfolio columns: ['reward', 'social', 'web', 'mobile', 'email', 'difficulty', 'duration', 'offer_type', 'promo_id']
transcript columns: ['customer_id', 'offer_id', 'event', 'amount', 'Reward', 'time']

Event counts:
event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64


In [67]:
def get_customer_offer_windows(customer_id):
    cust = transcript[transcript["customer_id"] == customer_id].copy()
    cust = cust.sort_values("time")

    received = cust[cust["event"] == "offer received"].copy()

    received = received.merge(
        Portfolio[["promo_id", "difficulty", "duration", "reward", "offer_type"]],
        left_on="offer_id",
        right_on="promo_id",
        how="left"
    )

    received["end_time"] = received["time"] + received["duration"] * 24

    result = []

    for i, row in received.reset_index(drop=True).iterrows():
        start = row["time"]
        end = row["end_time"]
        offer_id = row["offer_id"]

        window = cust[(cust["time"] >= start) & (cust["time"] <= end)].copy()

        window["offer_type"] = row["offer_type"]
        window["difficulty"] = row["difficulty"]

        result.append(window)
    result = pd.concat(result, ignore_index=True)
    result = result[[
    "customer_id",
    "offer_id",
    "offer_type",
    "difficulty",
    "event",
    "amount",
    "Reward",
    "time"
    ]]

    return result

    

In [ ]:
# 이상하다고 본 고객들

# suspicious_customers = [
# "157034648e3b422f8f7af11778bacb69",
# "1f6ad0a8def240b3ad633f83032dedd8",
# "27aa749a6f5f448e91327028a2ac7fb5",
# "2de68905470540e6a991f0763654d3c4",
# "3579056d94644a9481e05012b48fe252",
# "39437fa966ad4de0bc8afe8906778822",
# "3ead7d94371041fcab4db5fb57ced95c",
# "41aee69895c341108f1e4d65a711f8b9",
# "45e5d63c00cc41bd8d8d6f11e43b4d67",
# "46e3831f06254440a99bf4c371e378ff",
# "5ae36f912be1492199ec2da838cc6dda",
# "677b68c6df3449c2a2853d551c8b2ac3",
# "6c9b18ecc7054a3fa2e56d793a09f46b"
# ]

In [68]:
customer_windows = get_customer_offer_windows("157034648e3b422f8f7af11778bacb69")
display(customer_windows)

,customer_id,offer_id,offer_type,difficulty,event,amount,Reward,time
0,157034648e3b422f8f7af11778bacb69,5a8bc65990b245e5a138643cd4eb9837,informational,0,offer received,NaN,NaN,0
1,157034648e3b422f8f7af11778bacb69,5a8bc65990b245e5a138643cd4eb9837,informational,0,offer viewed,NaN,NaN,18
2,157034648e3b422f8f7af11778bacb69,2906b810c7d4411798c6938adc9daaa5,discount,10,offer received,NaN,NaN,168
3,157034648e3b422f8f7af11778bacb69,2906b810c7d4411798c6938adc9daaa5,discount,10,offer viewed,NaN,NaN,300
4,157034648e3b422f8f7af11778bacb69,2906b810c7d4411798c6938adc9daaa5,discount,10,offer completed,NaN,2.0,306
5,157034648e3b422f8f7af11778bacb69,NaN,discount,10,transaction,11.74,NaN,306
6,157034648e3b422f8f7af11778bacb69,3f207df678b143eea3cee63160fa8bed,informational,0,offer received,NaN,NaN,408
7,157034648e3b422f8f7af11778bacb69,NaN,informational,0,transaction,17.46,NaN,444
8,157034648e3b422f8f7af11778bacb69,NaN,informational,0,transaction,6.97,NaN,498
9,157034648e3b422f8f7af11778bacb69,ae264e3637204a6fb9bb56bc8210ddfd,informational,0,offer received,NaN,NaN,504
